# Lab 5D: Fresh-Run Inference and Drift Monitoring

This notebook creates one isolated monitoring run for the deployed bank-marketing endpoint. It generates shifted traffic, waits for exactly those captured rows, assigns simulated ground truth only to those rows, compares them with the model's registered baselines, and persists one internally consistent monitoring record.

The executable mechanics live in `src/drift_monitoring/monitoring_session.py`. The short cells below expose the sequence and the decisions without duplicating SQL, polling, or persistence code.

## Prerequisites

Run Lab 0 and the earlier data/model labs first. This notebook expects:

- the shared `.env` created by Lab 0;
- the `training_data`, `evaluation_data`, `inference_responses`, `monitoring_responses`, and `ground_truth_updates` Athena tables from the workshop stack;
- an approved model package with `baseline.json`; and
- a live `bank-marketing-*` endpoint with inference capture enabled.

Running this notebook invokes the endpoint and writes ground-truth and monitoring rows. It does not create infrastructure or change QuickSight resources.

## 1. Resolve the environment and monitoring session

`load_shared_environment()` finds the nearest ancestor `.env`. `MonitoringSession` then resolves the endpoint, AWS clients, schema, and thresholds from checked-in configuration.

In [ ]:
from src.config.notebook import ensure_lab5_package, load_shared_environment

env_path = load_shared_environment()
package_path = ensure_lab5_package()
from src.drift_monitoring.monitoring_session import MonitoringSession

session = MonitoringSession()
print(f'Environment: {env_path}')
print(f'Package: {package_path}')
session.summary()

### 1.1 Verify the monitoring tables

This is a read-only Glue metadata check. If a table is missing, redeploy the workshop stack instead of creating an ad hoc table from this notebook.

In [ ]:
table_status = session.verify_monitoring_tables()
table_status

if not all(table_status.values()):
    raise RuntimeError(f'Missing workshop table(s): {[k for k, v in table_status.items() if not v]}')

### 1.2 Resolve baseline lineage

The data-drift reference is the training snapshot registered with the model that serves the endpoint. The model-drift reference is the corresponding held-out evaluation snapshot, re-scored with that same model artifact. This avoids comparing current traffic with whichever model happened to train most recently.

In [ ]:
lineage = session.load_baseline()
lineage

## 2. Generate deterministic drift traffic

The generator samples independent rows from `training_data` and applies the configured shifts in `src/config/config.yaml`. The default scenario shifts five model features. The random seed and sample count are also configured there, so a rerun is reproducible.

In [ ]:
drift_csv = session.generate_drifted_data()
print(f'Generated: {drift_csv}')

## 3. Capture one fresh inference batch

The session asks Athena for the lower bound **before** invoking the endpoint. It sends 101 complete rows, polls until exactly 101 new capture rows exist, chooses the last captured request as the upper bound, and immediately tags those rows with a unique `monitoring_run_id`.

Every later query selects by that persisted tag. It never uses the previous monitoring row or a rolling lookback. If concurrent endpoint traffic makes the row count exceed 101, capture fails instead of silently mixing runs. Rows that arrive after the upper bound remain untagged and cannot enter this run.

In [ ]:
window = session.start_run()
print(f'Run: {window.monitoring_run_id}')
print(f'Lower bound (Athena UTC): {window.started_at}')

In [ ]:
traffic = session.send_drift_traffic(count=101, csv_path=drift_csv)
traffic

In [ ]:
capture = session.close_capture(timeout_seconds=240, poll_seconds=10)
capture

## 4. Simulate and apply ground truth for this batch

Production labels normally arrive from an outcome system. For this workshop, the simulator copies each prediction and flips a deterministic fraction of labels. It reads only rows tagged with this run ID and writes the same ID as `batch_id` in `ground_truth_updates`. The updater then merges only those inference IDs.

The returned coverage must equal the captured row count. This check prevents old unlabeled rows from being included and prevents a partial label batch from being graphed as complete.

In [ ]:
ground_truth = session.apply_simulated_ground_truth(seed=42)
ground_truth

## 5. Data drift

This notebook overrides Evidently's automatic test selection. It pins Jensen-Shannon distance for numeric and categorical columns in every new report. It uses a threshold of `0.1`.

### How to read drift scores

- **Jensen-Shannon distance (JSD):** `0` means the distributions are the same. A higher value means more drift.
- **Kolmogorov-Smirnov (KS):** This test compares numeric distributions. It returns a p-value. A lower p-value means stronger evidence of drift.
- **Normalized Wasserstein distance:** This measures how far a numeric distribution moves. A higher value means more drift.
- **Chi-square and Z-test:** Chi-square compares category frequencies. A Z-test compares binary proportions. Both return p-values. A lower p-value means stronger evidence of drift.

For the pinned Evidently 0.7.x family, automatic selection uses KS for numeric data with 1,000 or fewer observations. It uses normalized Wasserstein distance for numeric data with more than 1,000 observations. For categorical data with 1,000 or fewer observations, it uses Chi-square when there are more than two categories and Z-test for binary data. It uses Jensen-Shannon distance for categorical data with more than 1,000 observations. These are Evidently's general defaults. They are alternate behavior here because this notebook pins JSD instead.

Distances and p-values point in opposite directions. Higher distances indicate more drift. Lower p-values indicate more evidence of drift.

The charts therefore use normalized `drift_magnitude`. It accounts for each test's threshold and direction. `1.0` is the boundary. Values above `1.0` indicate drift, regardless of the raw-score direction. Sampling variation can move scores up or down between runs. Do not expect a strictly monotonic trend.

The dataset verdict uses a separate rule: `drifted_columns_share >= DATA_DRIFT_THRESHOLD` (default 20%). Only features with at least 50 non-null observations on both sides enter that denominator. All-null or undersampled features are persisted with null score, magnitude, and verdict rather than zero.

In [ ]:
import pandas as pd

data_drift = session.run_data_drift(baseline_limit=5000)
print({k: data_drift[k] for k in (
    'drift_detected', 'drifted_columns_count',
    'drifted_columns_share', 'features_analyzed', 'sample_size')})
pd.DataFrame(data_drift['per_column']).T.sort_values(
    'drift_magnitude', ascending=False, na_position='last')

## 6. Model drift

Current labels, predicted labels, and positive-class probabilities come from the same tagged inference rows and are cleaned together, preserving row pairing. ROC-AUC requires both ground-truth classes. Evidently's classification comparison additionally requires both predicted classes on each side.

If either requirement is unavailable, `skip_reason` is explicit and model fields remain null. Otherwise the run is model drift when relative ROC-AUC degradation reaches `MODEL_DRIFT_THRESHOLD` (default 5%).

In [ ]:
model_drift = session.run_model_drift(prediction_threshold=0.5)
{k: v for k, v in model_drift.items() if k != 'report'}

if model_drift['report']:
    display(model_drift['report']['snapshot'])
else:
    print(f"Model comparison skipped: {model_drift['skip_reason']}")

## 7. Persist and verify the run

One MLflow run receives the same data/model results, then one Athena monitoring row is inserted. `data_sample_size` must match tagged inference rows. When model metrics are available, `model_sample_size` must match labeled tagged rows. Per-feature JSON retains raw score, canonical magnitude, method, threshold, direction, sample counts, comparability, and unavailable reason.

In [ ]:
persisted = session.persist_results(log_to_mlflow=True)
persisted

## 8. Building a real trend

This branch has no independent batch simulator. Lab 5D therefore does not fabricate backdated monitoring rows. One complete execution contributes one real chart point. Re-run Sections 2 through 7 later to add another point; each execution receives a new run ID and analyzes only its own tagged rows.

Lab 5E uses `drift_magnitude` for aggregate feature charts, `drifted_columns_share` for dataset-share charts, and preserves unavailable values as null. Reference lines mark feature magnitude `1.0` and the configured dataset-share threshold.

## Teardown

This notebook creates no infrastructure. The generated CSV is local; inference, ground-truth, and monitoring rows belong to the workshop tables. Delete the endpoint from Lab 3A when all Lab 5 work is complete, then follow the workshop stack cleanup instructions.